In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import st_clustering as stc
import seaborn as sns
import os

In [2]:
np.random.seed(0)

In [3]:
df = pd.read_csv('z_matrix_after_ipca.csv')
df_with_time = df
df_without_time = df.iloc[:, 2:]

In [4]:
# Normalisation of ST data
df['ipca1'] = (df['ipca1'] - df['ipca1'].min()) / (df['ipca1'].max() - df['ipca1'].min())
df['ipca2'] = (df['ipca2'] - df['ipca2'].min()) / (df['ipca2'].max() - df['ipca2'].min())
# Transformation to numpy array
data_st = df.loc[:, ['time','ipca1', 'ipca2']].values

In [ ]:
# This is where ST Clustering happens. 
st_spectral = stc.ST_SpectralClustering(eps1=0.1,eps2=20,n_clusters=3)
st_spectral.st_fit(data_st)

D:\Conda\envs\stcluster\lib\site-packages\st_clustering\st_clustering.py:31: UserWarning: As input data is very large, clustering probably will take some time. Thus, if clustering takes too long, it might make sense to use st_fit_frame_split method.
  warnings.warn('As input data is very large, clustering probably will take some time. Thus, if clustering takes too long, it might make sense to use st_fit_frame_split method.')


In [ ]:
df['st_labels'] = st_spectral.labels

In [ ]:
# Incorporation of hopping events in our clustering results

df_hops = pd.read_csv("Hopping_Information.csv")
df_hops = df_hops[df_hops["time"] <= 57.8].reset_index(drop=True)

df_plot_st = df[df["time"] <= 57.8].reset_index(drop=True) 
###data_plot = df_plot.loc[:, ["ipca1", "ipca2"]].values
###labels_st = df_plot["st_labels"].values

hop_mask_st = (df_hops["Hops2.1"] == 1) | (df_hops["Hops1.2"] == 1)
df_plot_st["is_hop"] = hop_mask_st.values 

In [ ]:
df_zmt = pd.read_csv("Z_Matrix (raw).csv")
df_zmt = df_zmt.sort_values(["time", "traj"]).reset_index(drop=True)
df_zmt = df_zmt[["time", "traj"] + [c for c in df_zmt.columns if c not in ["time", "traj"]]]
df_zmt = df_zmt[df_zmt["time"] <= 57.8].reset_index(drop=True)

df_zmt_st = df_zmt.copy()
df_zmt_st["st_labels"] = df["st_labels"].values
df_zmt_st.head()

In [ ]:
# Normalisation of NonST data
df_without_time['ipca1'] = (df_without_time['ipca1'] - df_without_time['ipca1'].min()) / (df_without_time['ipca1'].max() - df_without_time['ipca1'].min())
df_without_time['ipca2'] = (df_without_time['ipca2'] - df_without_time['ipca2'].min()) / (df_without_time['ipca2'].max() - df_without_time['ipca2'].min())
# Transformation to numpy array
data_nonst = df_without_time.loc[:, ['ipca1', 'ipca2']].values

In [ ]:
# This is where NonST Clustering happens. 

from sklearn.cluster import SpectralClustering

spectral = SpectralClustering(
    n_clusters=3,
    assign_labels="discretize",
    affinity="nearest_neighbors",
    n_neighbors=10,
    random_state=0
)

labels = spectral.fit_predict(data_nonst)
df["nonst_labels"] = labels

In [ ]:
# Incorporation of Hopping Information

df_plot_nonst = df[df["time"] <= 57.8].reset_index(drop=True)
####data_plot = df_plot.loc[:, ["ipca1", "ipca2"]].values
####labels_nonst = df_plot["nonst_labels"].values

hop_mask_nonst = (df_hops["Hops2.1"] == 1) | (df_hops["Hops1.2"] == 1)
df_plot_nonst["is_hop"] = hop_mask_nonst.values

In [ ]:
df_zmt_nonst = df_zmt.copy()
df_zmt_nonst['nonst_labels'] = df['nonst_labels'].values
df_zmt_nonst.head()

In [ ]:
# Plot dihedral angles against C5-C6 bond distance 

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True)

# =========================
# Plot 1: ST
# =========================
ax = axes[0]

cluster_labels = sorted(df_zmt_st['st_labels'].unique())
colors = ['blue', 'gold', 'red']

idx_hops = df_hops[
    (df_hops['Hops2.1'] == 1) | (df_hops['Hops1.2'] == 1)
].index.tolist()

for l, c in zip(cluster_labels, colors):
    zmt = df_zmt_st[df_zmt_st['st_labels'] == l].abs()
    ax.scatter(
        zmt['r65'], zmt['d12654'],
        s=8, c=c, alpha=0.2,
        label=f'cluster {l}'
    )

    df_zmt_hops = df_zmt_st.loc[idx_hops]
    zmt = df_zmt_hops[df_zmt_hops['st_labels'] == l].abs()
    if not zmt.empty:
        ax.scatter(
            zmt['r65'], zmt['d12654'],
            s=180, marker='*',
            c=c, edgecolors='black',
            linewidths=1.2,
            alpha=0.9,
            zorder=5
        )

ax.set_title('ST Spectral Clustering')
ax.set_xlabel('C$_5$–C$_6$ Bond Distance ($\AA$)')
ax.set_ylabel('H$_{11}$–C$_6$–C$_5$–C$_4$ Angle ($^\circ$)')
ax.legend()


# =========================
# Plot 2: Non-ST
# =========================
ax = axes[1]

cluster_labels = sorted(df_zmt_nonst['nonst_labels'].unique())

for l, c in zip(cluster_labels, colors):
    zmt = df_zmt_nonst[df_zmt_nonst['nonst_labels'] == l].abs()
    ax.scatter(
        zmt['r65'], zmt['d12654'],
        s=8, c=c, alpha=0.2,
        label=f'cluster {l}'
    )

    df_zmt_hops = df_zmt_nonst.loc[idx_hops]
    zmt = df_zmt_hops[df_zmt_hops['nonst_labels'] == l].abs()
    if not zmt.empty:
        ax.scatter(
            zmt['r65'], zmt['d12654'],
            s=180, marker='*',
            c=c, edgecolors='black',
            linewidths=1.2,
            alpha=0.9,
            zorder=5
        )

ax.set_title('Non-ST Spectral Clustering')
ax.set_xlabel('C$_5$–C$_6$ Bond Distance ($\AA$)')
ax.set_ylabel('H$_{11}$–C$_6$–C$_5$–C$_4$ Angle ($^\circ$)')
ax.legend()

plt.tight_layout()
plt.savefig("Figure 11", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df_zmatrix_ipca_st = pd.read_csv("z_matrix_after_ipca.csv")
df_zmatrix_ipca_st["st_labels"] = df["st_labels"].values

In [ ]:
df_zmatrix_ipca_nonst = pd.read_csv("z_matrix_after_ipca.csv")
df_zmatrix_ipca_nonst["nonst_labels"] = df["nonst_labels"].values

In [ ]:
df_zmatrix_ipca_st.head()

In [ ]:
df_zmatrix_ipca_nonst.head()

In [ ]:
# Keeps the rows where hop_mask is True

df_zmatrix_ipca_st_hops = df_zmatrix_ipca_st.loc[hop_mask_st].reset_index(drop=True)
df_zmatrix_ipca_nonst_hops = df_zmatrix_ipca_nonst.loc[hop_mask_nonst].reset_index(drop=True)

In [ ]:
df_zmatrix_ipca_st_hops.head()

In [ ]:
df_zmatrix_ipca_nonst_hops.head()

In [ ]:
# Performing Violin plot analysis

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.violinplot(
    x="st_labels",
    y="time",
    data=df_zmatrix_ipca_st_hops,
    inner="box"
)
plt.xlabel("ST Cluster")
plt.ylabel("Time (fs)")
plt.title("Time Distribution per ST Cluster (Hops Only)")

plt.subplot(1, 2, 2)
sns.violinplot(
    x="nonst_labels",
    y="time",
    data=df_zmatrix_ipca_nonst_hops,
    inner="box"
)
plt.xlabel("NonST Cluster")
plt.ylabel("Time (fs)")
plt.title("Time Distribution per NonST Cluster (Hops Only)")

plt.tight_layout()
plt.savefig("Figure_12.png", dpi=300, bbox_inches="tight")
plt.show()
